In [1]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)

## Creating an entry in a vector DB

In [2]:
import psycopg2
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
db_user = os.getenv('POSTGRESQL_DEV_USER')
db_password = os.getenv('POSTGRESQL_DEV_PASSWORD')

In [14]:
conn = psycopg2.connect(
        host='aurora-postgresql-rag-instance-1.c0z0coi28tap.us-east-1.rds.amazonaws.com',
        dbname='documents',
        user=db_user,
        password=db_password
)

## Creating an embedding an adding it to database

In [24]:
from app.services.embeddings.bedrock_service import embed_text
import json

In [23]:
text = """
                        Cobertura del Servicio

                        La cobertura del servicio actualmente se limita a las regiones de Lima Metropolitana y La provincia del Callao, incluyendo los distritos de: 
                        
                        Callao
                        Ancon
                        Ate
                        Barranco
                        BreÃ±a
                        Carabayllo
                        Chaclacayo
                        Chorrillos
                        Cieneguilla
                        Comas
                        El Agustino
                        Independencia
                        JesÃºs MarÃ­a
                        La Molina
                        La Victoria
                        Lima
                        Lince
                        Los Olivos
                        Lurigancho
                        LurÃ­n
                        Magdalena del Mar
                        Miraflores
                        Pachacamac
                        Pucusana
                        Pueblo Libre
                        Puente Piedra
                        Punta Hermosa
                        Punta Negra
                        Rimac
                        San Bartolo
                        San Borja
                        San Isidro
                        San Juan de Lurigancho
                        San Juan de Miraflores
                        San Luis
                        San MartÃ­n de Porres
                        San Miguel
                        Santa Anita
                        Santa MarÃ­a del Mar
                        Santa Rosa
                        Santiago de Surco
                        Surquillo
                        Villa El Salvador
                        Villa MarÃ­a del Triunfo

                        Las demas regiones se encuentran en proceso de integración.
                        Para más información visite nuestra pagina: housy.pe
                         """
vector_text = embed_text(text)

In [33]:
conn = psycopg2.connect(
        host='aurora-postgresql-rag-instance-1.c0z0coi28tap.us-east-1.rds.amazonaws.com',
        dbname='documents',
        user=db_user,
        password=db_password
)

cur = conn.cursor()

#query
cur.execute("""
INSERT INTO faq_embeddings(document_name, page_number, chunk_index, metadata, content, embedding, s3_link)
VALUES (
        %s, %s, %s , %s, %s, %s ,%s )
""",(
    "cobertura.pdf",
    1,
    0,
    json.dumps({"data": "test"}),
    text,
    vector_text,
    "s3:random/link.com"
)
)

conn.commit()
cur.close()
conn.close()

## Vector Search

In [35]:
user_query = '¿Cuál es la cobertura del servicio?'
user_query_embed = embed_text(user_query)

In [38]:
conn = psycopg2.connect(
        host='aurora-postgresql-rag-instance-1.c0z0coi28tap.us-east-1.rds.amazonaws.com',
        dbname='documents',
        user=db_user,
        password=db_password
)

cur = conn.cursor()

cur.execute(
"""
SELECT content, metadata, s3_link
FROM faq_embeddings
ORDER BY embedding <-> %s::vector
LIMIT 3
""", (user_query_embed,)
)

results = cur.fetchall()

for row in results:
    print(row)


cur.close()
conn.close()

('\n                        Cobertura del Servicio\n\n                        La cobertura del servicio actualmente se limita a las regiones de Lima Metropolitana y La provincia del Callao, incluyendo los distritos de: \n                        \n                        Callao\n                        Ancon\n                        Ate\n                        Barranco\n                        BreÃ±a\n                        Carabayllo\n                        Chaclacayo\n                        Chorrillos\n                        Cieneguilla\n                        Comas\n                        El Agustino\n                        Independencia\n                        JesÃºs MarÃ\xada\n                        La Molina\n                        La Victoria\n                        Lima\n                        Lince\n                        Los Olivos\n                        Lurigancho\n                        LurÃ\xadn\n                        Magdalena del Mar\n                  

In [44]:
print(results[0][0])


                        Cobertura del Servicio

                        La cobertura del servicio actualmente se limita a las regiones de Lima Metropolitana y La provincia del Callao, incluyendo los distritos de: 
                        
                        Callao
                        Ancon
                        Ate
                        Barranco
                        BreÃ±a
                        Carabayllo
                        Chaclacayo
                        Chorrillos
                        Cieneguilla
                        Comas
                        El Agustino
                        Independencia
                        JesÃºs MarÃ­a
                        La Molina
                        La Victoria
                        Lima
                        Lince
                        Los Olivos
                        Lurigancho
                        LurÃ­n
                        Magdalena del Mar
                        Miraflores
                 

In [22]:
#function wrapping
def retrieve_entries(user_query: str):
    # conversion query 
    user_query = '¿Cuál es la cobertura del servicio?'
    user_query_embed = embed_text(user_query)

    conn = psycopg2.connect(
        host='aurora-postgresql-rag-instance-1.c0z0coi28tap.us-east-1.rds.amazonaws.com',
        dbname='documents',
        user=db_user,
        password=db_password
    )

    cur = conn.cursor()

    cur.execute(
    """
    SELECT content, metadata, s3_link
    FROM faq_embeddings
    ORDER BY embedding <-> %s::vector
    LIMIT 3
    """, (user_query_embed,)
    )

    results = cur.fetchall()

    cur.close()
    conn.close()

    return results

In [25]:
result = retrieve_entries("Cual es la cobertura del servicio")

In [26]:
result

[('\n                        Cobertura del Servicio\n\n                        La cobertura del servicio actualmente se limita a las regiones de Lima Metropolitana y La provincia del Callao, incluyendo los distritos de: \n                        \n                        Callao\n                        Ancon\n                        Ate\n                        Barranco\n                        BreÃ±a\n                        Carabayllo\n                        Chaclacayo\n                        Chorrillos\n                        Cieneguilla\n                        Comas\n                        El Agustino\n                        Independencia\n                        JesÃºs MarÃ\xada\n                        La Molina\n                        La Victoria\n                        Lima\n                        Lince\n                        Los Olivos\n                        Lurigancho\n                        LurÃ\xadn\n                        Magdalena del Mar\n                 

In [34]:
def format_context(context: list):
    context_string = ""
    for item in context:
        context_string += item[0] + "\n"
    return context_string

In [36]:
formatted_context = format_context(result)

### Langchain Integration

In [ ]:
from app.core.aws_clients import get_langchain_bedrock_client
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
faq_prompt = """
You are a useful assistant who answers user queries based on the context provided below. Provide concise answers in a gently manner, and respond in the same language as the user
If you don't find the required information in the context, reply simply with: 'I don't have that information, sorry' 

Context:
{context}
"""

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", faq_prompt),
        ("human", "{user_input}")
    ]
)

In [16]:
llm = get_langchain_bedrock_client(model_id="amazon.nova-lite-v1:0")

In [17]:
chain = prompt | llm

In [39]:
chat_response = chain.invoke({
    'context': formatted_context,
    'user_input': 'Existe cobertura en Arequipa?'
})

In [40]:
chat_response

AIMessage(content='No, actualmente la cobertura del servicio se limita a las regiones de Lima Metropolitana y La provincia del Callao. Arequipa no está incluida en la cobertura actual.', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '976bcba6-0f61-4ead-9ad4-4d1c0ba6f697', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 20 Sep 2025 03:34:21 GMT', 'content-type': 'application/json', 'content-length': '370', 'connection': 'keep-alive', 'x-amzn-requestid': '976bcba6-0f61-4ead-9ad4-4d1c0ba6f697'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [327]}, 'model_name': 'amazon.nova-lite-v1:0'}, id='run--96bde4e5-94ef-4b99-85c6-32dc2da0db4e-0', usage_metadata={'input_tokens': 375, 'output_tokens': 32, 'total_tokens': 407, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})